## {PATTERN_NAME} Knowledge Graph Building Content

This notebook follows the MaaS indexing workflow: it discovers documents in S3, extracts text with Docling, chunks the documents, and creates a collection-scoped Neo4j knowledge graph. MaaS supplies embeddings and entity/relation extraction. Run this notebook before the graph inference notebook.

### &#x1F4CB; Contents 
This notebook contains the following sections:

- **[Setup](#Setup)**
  - [Install packages](#Install-packages)
  - [Prerequisites for Disconnected Clusters (Optional)](#Prerequisites-for-Disconnected-Clusters)
  - [Configure Models for Disconnected Environments (Optional)](#Configure-Models-for-Disconnected-Environments)
  - [Import required libraries](#Import-required-libraries)
  - [Configure S3 credentials](#Configure-S3-credentials)
  - [Prepare S3 client](#Prepare-S3-client)
- **[Process input documents](#Process-input-documents)**
  - [Documents discovery](#Documents-discovery)
  - [Select required extraction models](#Select-required-extraction-models)
  - [Download OCR models for connected clusters](#Download-OCR-models-for-connected-clusters)
  - [Validate extraction models and configure OCR](#Validate-extraction-models-and-configure-OCR)
  - [Text extraction](#Text-extraction)
- **[Upload documents content into vector store](#Upload-documents-content-into-vector-store)**
  - [Prepare MaaS client](#Prepare-MaaS-client)
  - [Prepare chunker](#Prepare-chunker)
  - [Initialize vector store](#Initialize-vector-store)
  - [Upload chunks to vector store](#Upload-chunks-to-vector-store)
  - [Retrieve chunks for sample question](#Retrieve-chunks-for-sample-question)
- **[Summary](#Summary)**
- [Appendix: Downloading Models for Offline Use](#Appendix:-Downloading-Models-for-Offline-Use)

---

## Setup

This section sets up the notebook environment by installing required packages, importing libraries, and configuring access to S3 storage.

### Install packages

Install all required Python packages for document processing and RAG operations:
- **ai4rag**: The AutoRAG framework for building RAG applications

In [ ]:
%pip install 'ai4rag[text-extraction]~={AI4RAG_VERSION}' nest-asyncio | tail -n 1

### Prerequisites for Disconnected Clusters

If you are running this notebook on a **disconnected cluster** (no internet access), you must pre-download the required ML models before execution.

**Why:** This notebook selects its extraction backends after **Documents discovery**. PDF/image inputs require the pipeline-compatible PP-OCRv4 bundle. Disconnected runs must mount the bundle or use a workbench image that includes it; connected runs download the pinned OCR files when no local bundle is configured. Audio inputs require a complete local Whisper model. Text-only inputs require neither bundle.

**What you need:**
- PDF/image data: **Docling/RapidOCR artifacts** (core + RapidOCR models) — ~300-400 MB, exposed through `DOCLING_ARTIFACTS_PATH`
- Audio data: approved Hugging Face **Whisper** model, exposed through `HF_MODEL_DIR`
- Text-only data: neither OCR nor Whisper artifacts are required

**Environment variables:**
- `DOCLING_ARTIFACTS_PATH`: Path to pre-downloaded Docling/RapidOCR artifacts, required only for PDF/image data
- `HF_MODEL_DIR`: Path to the approved local Hugging Face Whisper model, required only for audio data

#### Trust private certificates with DSCI

If a disconnected-cluster endpoint (for example, an internal PyPI mirror, S3-compatible object store, or MaaS endpoint) uses a private or self-signed CA, add its CA certificate to the OpenShift AI trust bundle. Do not set `verify=False` or use an unverified client.

1. Create or update a ConfigMap containing the PEM CA bundle in the OpenShift AI applications namespace. A single `ca-bundle.crt` file may contain multiple PEM certificates.

   ```sh
   oc -n redhat-ods-applications create configmap custom-ca-bundle \
     --from-file=ca-bundle.crt=/path/to/ca-bundle.crt \
     --dry-run=client -o yaml | oc apply -f -
   ```

2. Reference that ConfigMap from the cluster `DSCInitialization` (DSCI) object. Replace `default-dsci` if your installation uses a different DSCI name.

   ```sh
   oc patch dscinitialization default-dsci --type=merge \
     -p '{{"spec":{{"trustedCABundle":{{"customCABundle":"custom-ca-bundle"}}}}}}'
   ```

OpenShift AI automatically propagates this custom CA into the managed trust bundles consumed by workbenches and other OpenShift AI services that mount the custom CA bundle. Restart affected workbenches after the bundle changes. This does not add trust to arbitrary workloads that do not consume the OpenShift AI custom CA bundle.

The notebook uses the workbench bundle at `/etc/pki/tls/custom-certs/ca-bundle.crt` when it is mounted. To use a bundle mounted at a different path, set `AWS_CA_BUNDLE` to that PEM bundle path before running the S3 setup cell; the S3 client uses it while retaining TLS certificate validation.

👉 This configuration is required for every workbench, including those with internet access.

👉 **For setup instructions, see [Appendix: Downloading Models for Offline Use](#Appendix-Downloading-Models-for-Offline-Use) at the end of this notebook.**

### Configure Models for Disconnected Environments

Run code cells tagged `disconnected` only on disconnected clusters. They configure offline execution; the graph notebook also reinstalls the tested OpenCV build from the internal package index. AutoX selects these cells with its explicit notebook cluster mode.

Model selection and validation run after Documents discovery in **both environments**. Disconnected PDF/image runs require `DOCLING_ARTIFACTS_PATH`; connected runs can use the cell tagged `connected` to download the pinned OCR files. Audio inputs require `HF_MODEL_DIR`. Text-only inputs require neither.

In [ ]:
# Disconnected clusters: install the tested headless OpenCV build from the internal package index.
%pip install --force-reinstall --no-cache-dir opencv-python-headless==4.14.0.94 | tail -n 1

In [ ]:
import os

# Run this cell only in disconnected environments; AutoX uses its disconnected tag.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

### Import required libraries

Import all necessary Python modules and configure logging to suppress verbose output from component loggers.

In [ ]:
import getpass
import json
import logging
import nest_asyncio
import os
import warnings
from pathlib import Path

nest_asyncio.apply()

warnings.filterwarnings("ignore")

for logger_name in (
    "httpx",
    "documents-discovery",
    "text-extraction",
):
    logging.getLogger(logger_name).propagate = False

### Configure S3 credentials

To load documents from S3-compatible object storage, you need to provide credentials. If you're using OpenShift AI, these can be configured as data connections.

&#x1F4CC; **Action**: Provide the credentials for your S3 instance if they are not already set in the notebook environment.

&#x1F4A1; **Tip**: In the project, open **Connections** and add an **S3 compatible object storage connection** to a bucket you will use for documents and test data. Open **Workbenches**, edit your workbench, and attach the S3 connection you created so the notebook can read from the bucket. Save and restart the workbench if prompted.

In [ ]:
required_vars = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_S3_BUCKET"]
missing = [var for var in required_vars if not os.environ.get(var)]
if missing:
    raise ValueError(f"Missing required environment variables: {{missing}}")

### Prepare S3 client

Creates an S3 client session using the provided credentials. This client will be used to discover and download documents from the specified S3 bucket.

In [ ]:
from ai4rag.utils.clients.s3 import create_s3_client

workbench_ca_bundle = Path("/etc/pki/tls/custom-certs/ca-bundle.crt")
ssl_cert_path = str(workbench_ca_bundle) if workbench_ca_bundle.is_file() else None
if ssl_cert_path:
    os.environ.setdefault("AWS_CA_BUNDLE", ssl_cert_path)

effective_ssl_cert_path = ssl_cert_path or os.getenv("AWS_CA_BUNDLE")
s3_client = create_s3_client(verify=effective_ssl_cert_path or True)
s3_client.head_bucket(Bucket=os.environ["AWS_S3_BUCKET"])
print("S3 connection verified with TLS certificate validation enabled.")

---

## Process input documents

This section handles document discovery and text extraction. Documents are first discovered in S3 storage, then their content is extracted and saved as DoclingDocument JSON files for further processing.

The data processing pipeline prepares documents for the RAG system in multiple steps. Each step produces outputs stored under `step_outputs/`. 

| Step | Function | Purpose |
|------|----------|---------| 
| 1 | **`discover_documents`** | List documents under every selected prefix, prioritize benchmark-referenced docs, apply a size cap, and write a JSON manifest (no content download). |
| 2 | **`extract_text`** | Download the listed documents from S3 and extract text to DoclingDocument JSON files using Docling. |

In [ ]:
from ai4rag.utils.data.documents_discovery import discover_documents
from ai4rag.utils.data.test_data_loader import load_test_data
from ai4rag.utils.data.text_extraction import DoclingExtractionConfig, extract_text

step_output_dir = Path("./step_outputs")
input_data_bucket_name = os.environ["AWS_S3_BUCKET"]
input_data_keys = {INPUT_DATA_KEYS}
test_data_key = "{TEST_DATA_KEY}"
step_output_dir.mkdir(parents=True, exist_ok=True)

### Documents discovery

Lists objects in the S3 input bucket, filters by supported extensions (e.g., `.pdf`, `.docx`, `.pptx`, `.md`, `.html`, `.txt`), and builds a document set. Documents referenced in the benchmark are prioritized, then others are added until a configurable size limit (1 GB by default) is reached. This step does not download document contents but writes a JSON manifest (`documents_descriptor.json`) containing the bucket, prefixes, and list of selected object keys and sizes for the next step.

In [ ]:
import inspect

test_data_result = load_test_data(
    bucket_name=input_data_bucket_name,
    key=test_data_key,
    s3_client=s3_client,
)
test_data_doc_names = [
    document_key
    for record in test_data_result.data
    for document_key in record["correct_answer_document_keys"]
]

discovery_kwargs = {{
    "bucket_name": input_data_bucket_name,
    "test_data_doc_names": test_data_doc_names,
    "s3_client": s3_client,
}}
if "prefixes" in inspect.signature(discover_documents).parameters:
    discovery_kwargs["prefixes"] = input_data_keys
else:
    if len(input_data_keys) > 1:
        raise RuntimeError(
            "This notebook was generated with multiple input locations, but the installed ai4rag "
            "does not support them. Install ai4rag >= 0.18.0."
        )
    discovery_kwargs["prefix"] = input_data_keys[0] if input_data_keys else ""

result = discover_documents(**discovery_kwargs)
result.save(step_output_dir / "discovered_documents")

print(json.dumps(result.to_dict(), indent=4, ensure_ascii=False))

### Select required extraction models

PDF/image inputs enable OCR; audio inputs use the approved local Whisper model. Mixed corpora require both; text-only corpora require neither. The OCR language starts from the pattern's detected language and can be overridden below when the document language differs.

In [ ]:
discovered_documents = result.to_dict()["documents"]
ocr_extensions = {{".pdf", ".jpg", ".jpeg", ".png", ".tif", ".tiff"}}
audio_extensions = {{".wav", ".mp3", ".m4a", ".aac", ".ogg", ".flac"}}
discovered_extensions = {{Path(document["key"]).suffix.lower() for document in discovered_documents}}
needs_ocr = bool(discovered_extensions & ocr_extensions)
needs_asr = bool(discovered_extensions & audio_extensions)

print(f"Discovered extensions: {{sorted(discovered_extensions)}}")
print(f"OCR required: {{needs_ocr}}; Whisper required: {{needs_asr}}")

# Match the language selected for the indexing pipeline. Override this when the
# document language differs from the benchmark language, e.g. ocr_language = "zh".
pattern_language = {LANGUAGE} or {{}}
ocr_language = pattern_language.get("code") or pattern_language.get("name") or "english"
docling_artifacts_path = os.getenv("DOCLING_ARTIFACTS_PATH") if needs_ocr else None

asr_model_path = None
if needs_asr:
    asr_model_path = os.getenv("HF_MODEL_DIR")
    model_dir = Path(asr_model_path) if asr_model_path else None
    config_path = model_dir / "config.json" if model_dir is not None else None
    if model_dir is None or not model_dir.is_dir() or config_path is None or not config_path.is_file():
        raise FileNotFoundError(
            "Audio data requires HF_MODEL_DIR to point to a local Transformers model directory containing config.json."
        )
    try:
        with config_path.open(encoding="utf-8") as config_file:
            model_config = json.load(config_file)
    except (OSError, json.JSONDecodeError) as exc:
        raise ValueError("Audio data requires HF_MODEL_DIR/config.json to contain valid JSON.") from exc
    model_type = model_config.get("model_type", "") if isinstance(model_config, dict) else ""
    if model_type != "whisper":
        raise ValueError(f"Audio data supports only Hugging Face Whisper models; found model_type={{model_type!r}}.")
    required_model_files = ("preprocessor_config.json", "tokenizer_config.json")
    has_weights = any(
        (model_dir / name).is_file()
        for name in (
            "model.safetensors",
            "pytorch_model.bin",
            "model.safetensors.index.json",
            "pytorch_model.bin.index.json",
        )
    )
    if any(not (model_dir / name).is_file() for name in required_model_files) or not has_weights:
        raise FileNotFoundError(
            "Audio data requires HF_MODEL_DIR to point to a complete approved local Hugging Face Whisper model."
        )

In [ ]:
# Pin the same nested PP-OCRv4 bundle as the documents-indexing pipeline.
chinese_aliases = {{"chinese", "ch", "zh", "zho", "chi", "zh-cn", "zh-tw"}}
ocr_bundle = "chinese" if ocr_language.strip().lower() in chinese_aliases else "english"
ocr_model_paths = {{}}
if needs_ocr:
    rapidocr_bundles = {{
        "english": {{
            "ocr_det_model_path": "onnx/PP-OCRv4/det/en_PP-OCRv3_det_mobile.onnx",
            "ocr_cls_model_path": "onnx/PP-OCRv4/cls/ch_ppocr_mobile_v2.0_cls_mobile.onnx",
            "ocr_rec_model_path": "onnx/PP-OCRv4/rec/en_PP-OCRv4_rec_mobile.onnx",
            "ocr_rec_keys_path": "paddle/PP-OCRv4/rec/en_PP-OCRv4_rec_mobile/en_dict.txt",
        }},
        "chinese": {{
            "ocr_det_model_path": "onnx/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.onnx",
            "ocr_cls_model_path": "onnx/PP-OCRv4/cls/ch_ppocr_mobile_v2.0_cls_mobile.onnx",
            "ocr_rec_model_path": "onnx/PP-OCRv4/rec/ch_PP-OCRv4_rec_mobile.onnx",
            "ocr_rec_keys_path": "paddle/PP-OCRv4/rec/ch_PP-OCRv4_rec_mobile/ppocr_keys_v1.txt",
        }},
    }}
    if docling_artifacts_path:
        ocr_root = Path(docling_artifacts_path).expanduser().resolve() / "RapidOcr"
        ocr_model_paths = {{key: str(ocr_root / relative) for key, relative in rapidocr_bundles[ocr_bundle].items()}}


### Download OCR models for connected clusters

Run the cell tagged `connected` only when internet access is available. If PDF/image inputs need OCR and `DOCLING_ARTIFACTS_PATH` is unset, it downloads only the selected language's four pinned PP-OCRv4 files from the public Hugging Face mirror used by the pipeline image. Files are cached and passed directly to OCR; core Docling models use Docling's normal connected downloader.

A configured `DOCLING_ARTIFACTS_PATH` takes precedence and is validated below without downloading replacements. Disconnected runs skip this cell and require the staged bundle. Text-only inputs download no OCR models.


In [ ]:
# Download only on connected clusters when no staged Docling bundle is configured.
if needs_ocr and not docling_artifacts_path:
    from huggingface_hub import hf_hub_download

    # Same public artifact mirror as the pipeline image, pinned to a fixed snapshot.
    ocr_model_paths = {{
        key: hf_hub_download(
            repo_id="superlinked/docling-artifacts",
            filename=f"RapidOcr/{{relative}}",
            revision="310479be56f3a99a0fea75d930e11eea38bf5ef7",
        )
        for key, relative in rapidocr_bundles[ocr_bundle].items()
    }}


### Validate extraction models and configure OCR

Run this shared cell in **connected and disconnected environments**. PDF/image inputs use the same PP-OCRv4 files as the indexing pipeline, either staged locally or downloaded by the connected-only cell. Every selected detection, classification, recognition, and character-dictionary file is validated and pinned explicitly; no alternative OCR models are downloaded. Text-only inputs skip OCR artifact checks.

In [ ]:
if needs_ocr:
    if not ocr_model_paths:
        raise EnvironmentError(
            "PDF or image data requires DOCLING_ARTIFACTS_PATH on disconnected clusters. "
            "On connected clusters, run the connected OCR model download cell first. "
            "Use a workbench image with the Docling artifacts bundle or mount the pipeline's PP-OCRv4 bundle."
        )
    missing = sorted(path for path in ocr_model_paths.values() if not Path(path).is_file())
    if missing:
        raise FileNotFoundError(f"RapidOCR {{ocr_bundle}} models are missing:\n  - " + "\n  - ".join(missing))
    print(f"OCR language {{ocr_language!r}}: using the {{ocr_bundle}} PP-OCRv4 bundle")

docling_config = DoclingExtractionConfig(
    do_ocr=needs_ocr,
    ocr_lang=ocr_bundle,
    asr_model_path=asr_model_path,
    **ocr_model_paths,
)


### Text extraction

Reads the `documents_descriptor.json` produced by the discovery step, downloads each listed document from S3 into a temporary directory, and runs **Docling** to extract text. Output is one DoclingDocument JSON file per document (e.g., `report.pdf.json`) written to the artifact output path. Each document's `name` field preserves the original filename including its suffix. These files form the final text corpus for the RAG system.

In [ ]:
extracted_text_dir = step_output_dir / "extracted_text"
try:
    extraction_result = extract_text(
        documents=discovered_documents,
        bucket=result.bucket, output_dir=extracted_text_dir,
        s3_client=s3_client,
        docling_artifacts_path=docling_artifacts_path,
        docling_config=docling_config,
    )
except RuntimeError as exc:
    if "CERTIFICATE_VERIFY_FAILED" not in str(exc):
        raise
    raise RuntimeError(
        f"S3 TLS certificate validation failed. Verify that ssl_cert_path={{{{effective_ssl_cert_path!r}}}} "
        "is a PEM CA bundle that trusts the S3 endpoint. Do not disable certificate verification."
    ) from exc

print(
    f"Extracted {{extraction_result.processed_count}}/{{extraction_result.total_documents}} documents "
    f"({{extraction_result.error_count}} errors)"
)

---

## Build the Neo4j knowledge graph

This section uses the documents extracted above to create a collection-scoped Neo4j knowledge graph.

### Prepare MaaS client

OpenShift AI Models-as-a-Service (MaaS) serves every model from a single OpenAI-compatible endpoint. This section builds one client from the MaaS base URL and API key; the embedding model reuses it.

**Prerequisites:**
- `MAAS_API_KEY`: Your authentication key for the MaaS API
- `MAAS_BASE_URL`: The complete OpenAI-compatible endpoint URL, used verbatim (e.g. `https://<host>/v1`)

&#x1F4A1; **Tip**: In OpenShift AI Workbench, you can add these as environment variables or data connections to avoid entering them manually each time.

In [ ]:
from ai4rag.utils.clients.maas_client import create_maas_client

MAAS_API_KEY = os.getenv("MAAS_API_KEY") or getpass.getpass("Please enter 'MAAS_API_KEY': ")
MAAS_BASE_URL = os.getenv("MAAS_BASE_URL") or getpass.getpass("Please enter 'MAAS_BASE_URL': ")

client = create_maas_client(base_url=MAAS_BASE_URL, api_key=MAAS_API_KEY)

### Prepare chunker

The chunker splits extracted documents into smaller chunks for more effective retrieval. Configuration includes:
- **Chunking Method**: The algorithm used to split text (e.g., recursive character splitting)
- **Chunk Size**: Maximum number of characters per chunk
- **Chunk Overlap**: Number of overlapping characters between consecutive chunks to preserve context

Proper chunking ensures that retrieved context is both relevant and fits within the model's context window.

In [ ]:
from ai4rag.rag.chunking import LangChainChunker, DoclingChunker

chunking_method = "{CHUNKING_METHOD}"
chunk_size = {CHUNK_SIZE}
chunk_overlap = {CHUNK_OVERLAP}

if chunking_method == "hybrid":
    chunker = DoclingChunker(max_tokens=chunk_size)
else:
    chunker = LangChainChunker(method=chunking_method, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

## Initialize the graph store

Use the same collection name in the inference notebook. Neo4j graph retrieval is collection-isolated and supports `search_mode="graph"` only.

In [ ]:
from ai4rag.rag.embedding.openai_model import OpenAIEmbeddingModel, OpenAIEmbeddingParams
from ai4rag.rag.foundation_models.openai_model import OpenAIFoundationModel
from ai4rag.rag.vector_store.config import Neo4jConfig
from ai4rag.rag.vector_store.neo4j import Neo4jGraphStore

embedding_model = OpenAIEmbeddingModel(
    client=client,
    model_id="{EMBEDDING_MODEL_ID}",
    params=OpenAIEmbeddingParams(**{EMBEDDING_PARAMS}),
)
foundation_model = OpenAIFoundationModel(
    client=client,
    model_id="{FM_MODEL_ID}",
    params={FM_PARAMS},
)
collection_name = "{COLLECTION_NAME}"
vector_store = Neo4jGraphStore(
    embedding_model=embedding_model,
    config=Neo4jConfig.from_env(),
    collection_name=collection_name,
    foundation_model=foundation_model,
    kg_extraction_config={KG_EXTRACTION_CONFIG},
)

### Upload chunks to the knowledge graph

Each chunk is embedded, stored, and used to extract entities and relationships in the selected Neo4j collection.

In [ ]:
from ai4rag.utils.docling_io import load_docling_documents

documents = load_docling_documents(extracted_text_dir)

for document in documents:
    chunked_documents = chunker.split_documents([document])
    vector_store.add_documents(chunked_documents)

### Retrieve graph contexts for a sample question

In [ ]:
from dataclasses import asdict
from pprint import pprint

sample_question = input()
results = vector_store.search(
    query=sample_question,
    k=5,
    search_mode="graph",
    **{GRAPH_RETRIEVAL_CONFIG},
)
for chunk in results:
    pprint(asdict(chunk), indent=4)

---

## Summary

This notebook used the MaaS indexing workflow to discover and extract documents, then built a Neo4j knowledge graph that is ready for graph retrieval.

In [ ]:
vector_store.close()

---

## Appendix: Downloading Models for Offline Use

Use the same approved Docling/RapidOCR artifacts bundle as the documents-indexing pipeline. Stage it on an internet-connected machine before transferring it to the workbench, or use the workbench image that already contains it. A generic Docling model download may fetch a different OCR version and does not establish PP-OCRv4 parity.

### Step 1: Prepare the approved model bundle

The bundle must include the core Docling artifacts and the selected language's RapidOCR files below `RapidOcr/`:

| Model | English bundle | Chinese bundle |
|---|---|---|
| Detection | `onnx/PP-OCRv4/det/en_PP-OCRv3_det_mobile.onnx` | `onnx/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.onnx` |
| Classification | `onnx/PP-OCRv4/cls/ch_ppocr_mobile_v2.0_cls_mobile.onnx` | Same file |
| Recognition | `onnx/PP-OCRv4/rec/en_PP-OCRv4_rec_mobile.onnx` | `onnx/PP-OCRv4/rec/ch_PP-OCRv4_rec_mobile.onnx` |
| Character dictionary | `paddle/PP-OCRv4/rec/en_PP-OCRv4_rec_mobile/en_dict.txt` | `paddle/PP-OCRv4/rec/ch_PP-OCRv4_rec_mobile/ppocr_keys_v1.txt` |

### Step 2: Transfer Artifacts to the Disconnected Cluster

Copy the prepared bundle while preserving its directory structure, or mount the same model volume used by the pipeline:

```bash
rsync -av /path/to/approved/docling/artifacts/ user@cluster:/offline/models/docling/artifacts/
```

### Step 3: Configure and run

```bash
export DOCLING_ARTIFACTS_PATH=/offline/models/docling/artifacts
# Only when audio documents are selected:
export HF_MODEL_DIR=/models/whisper
```

Connected workbenches can run the cell tagged `connected` to download the same PP-OCRv4 files when `DOCLING_ARTIFACTS_PATH` is unset. Set the OCR language override when the document language differs from the pattern's benchmark language. The shared validation cell lists any missing files before extraction starts.